## ClavaDDPM Training

In [1]:
from pathlib import Path
import os

# Make sure we are in the root directory
def set_project_root(marker="pyproject.toml"):
    path = Path.cwd()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            os.chdir(parent)
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")
set_project_root()

PosixPath('/Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp')

In [2]:
import pickle
from logging import INFO
from pathlib import Path

from hydra import initialize, compose
from omegaconf import OmegaConf

from midst_toolkit.common.config import ClavaDDPMClassifierConfig, ClavaDDPMClusteringConfig, ClavaDDPMDiffusionConfig
from midst_toolkit.common.logger import TOOLKIT_LOGGER, log
from midst_toolkit.common.variables import DEVICE
from midst_toolkit.models.clavaddpm.clustering import clava_clustering
from midst_toolkit.models.clavaddpm.data_loaders import Table, load_tables
from midst_toolkit.models.clavaddpm.train import ClavaDDPMModelArtifacts, clava_training
# Preventing some excessive logging
TOOLKIT_LOGGER.setLevel(INFO)

## Set the paths and load the hydra config

In [4]:
ROOT = Path.cwd()
IMPLEMENTATION_ROOT = ROOT / "implementations" / "tabular_data"
print("IMPLEMENTATION ROOT: ", IMPLEMENTATION_ROOT)
# Set data and output directories
# Berka dataset: https://webpages.charlotte.edu/mirsad/itcs6265/group1/domain.html
base_data_dir = IMPLEMENTATION_ROOT / "multi_table" / "data" / "berka"
base_output_dir = IMPLEMENTATION_ROOT / "multi_table" / "results"

IMPLEMENTATION ROOT:  /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/tabular_data


In [6]:
# Context manager ensures global state is cleaned up after initialization
with initialize(version_base=None, config_path="."):
    # Load config.yaml
    cfg = compose(config_name="config")

# View the configuration as a standard YAML string
print(OmegaConf.to_yaml(cfg))

diffusion_config:
  d_layers:
  - 512
  - 512
  dropout: 0.0
  num_timesteps: 10
  model_type: mlp
  iterations: 10
  batch_size: 4096
  lr: 0.0006
  gaussian_loss_type: mse
  weight_decay: 1.0e-05
  scheduler: cosine
clustering_config:
  parent_scale: 1.0
  num_clusters: 50
  clustering_method: kmeans_and_gmm
classifier_config:
  d_layers:
  - 128
  - 128
  lr: 0.0001
  dim_t: 128
  batch_size: 4096
  iterations: 10



### Step 1: Load the tables

In [8]:
# Preventing some excessive logging
TOOLKIT_LOGGER.setLevel(INFO)
# Note: Use all the data for training. `load_tables` data split is only used to gather info.
# No actual train/test split is done here.
# `load_tables` loads tables f"{table}.csv" saved under `base_data_dir` and their domain files as f"{table}_domain.json".
log(INFO, f"Loading data from {base_data_dir}...")
tables, relation_order, _ = load_tables(Path(base_data_dir))

INFO :      Loading data from /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/tabular_data/multi_table/data/berka...
INFO :      Training data ratio is 1, so the data will not be split into training and test sets.
INFO :      Train dataframe shape: (77, 15)
INFO :      Total dataframe shape: (77, 15)
INFO :      Numerical data shape: (77, 13)
INFO :      Categorical data shape: (77, 2)
INFO :      Training data ratio is 1, so the data will not be split into training and test sets.
INFO :      Train dataframe shape: (5369, 4)
INFO :      Total dataframe shape: (5369, 4)
INFO :      Numerical data shape: (5369, 0)
INFO :      Categorical data shape: (5369, 4)
INFO :      Training data ratio is 1, so the data will not be split into training and test sets.
INFO :      Train dataframe shape: (4500, 2)
INFO :      Total dataframe shape: (4500, 2)
INFO :      Numerical data shape: (4500, 1)
INFO :      Categorical data shape: (4500, 1)
INFO :      Training

## Step 2: Cluster the data
If there is a clustering checkpoint saved at `base_output_dir/clustering_ckpt.pkl`, the `clava_clustering` will load and use it. Remember to remove old clustering checkpoints.

In [10]:
log(INFO, "Clustering step...")
clustering_config = ClavaDDPMClusteringConfig(**cfg.clustering_config)
tables, _ = clava_clustering(tables, relation_order, Path(base_output_dir), clustering_config)
log(INFO, "Clustering checkpoint is saved at `base_output_dir/clustering_ckpt.pkl`.")

INFO :      Clustering step...
INFO :      Clustering checkpoint found, loading...
INFO :      Clustering checkpoint is saved at `base_output_dir/clustering_ckpt.pkl`.


## Step 3: Train the model

In [12]:
log(INFO, "Training model...")
diffusion_config = ClavaDDPMDiffusionConfig(**cfg.diffusion_config)
classifier_config = ClavaDDPMClassifierConfig(**cfg.classifier_config)

tables, _ = clava_training(
    tables,
    relation_order,
    Path(base_output_dir),
    diffusion_config,
    classifier_config,
    device=DEVICE,
)
log(INFO, "Model trained successfully.")

INFO :      Training model...
INFO :      Training None -> district model from scratch
INFO :      No cache_dir provided. Will not attempt to load or save transformed dataset from/to cache
INFO :      No NaN processing policy specified.
INFO :      Model params: ModelParameters(diffusion_parameters=DiffusionParameters(layers_dimensions=[512, 512], dropout=0.0, input_dimension=0, output_dimension=0, embedding_dimension=0, n_blocks=0, block_dimension=0, hidden_dimension=0, dropout_first=0, dropout_second=0), input_dimension=np.int64(17), num_classes=0, is_target_conditioned=<IsTargetConditioned.NONE: 'none'>)
INFO :      Getting model: mlp
INFO :      Creating /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/tabular_data/multi_table/results/models. Saving None -> district model to /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/tabular_data/multi_table/results/models/None_district_ckpt.pkl
INFO :      Training distr

## Step 4: Load  the cluster checkpoint and the trained models

In [12]:
clustering_results_file = Path(base_output_dir) / "cluster_ckpt.pkl"
with open(clustering_results_file, "rb") as f:
    clustering_result = pickle.load(f)

assert all(isinstance(table, Table) for table in clustering_result["tables"].values())
assert isinstance(clustering_result["all_group_lengths_prob_dicts"], dict)
log(INFO, "Clustering checkpoint is loaded successfully.")


INFO :      Clustering checkpoint is loaded successfully.


In [13]:
for relation in relation_order:
    results_file = Path(base_output_dir) / "models" / f"{relation[0]}_{relation[1]}_ckpt.pkl"
    log(INFO, f"Checking the results from {results_file}...")

    with open(results_file, "rb") as f:
        result = pickle.load(f)

    # Asserting the results are the correct type
    assert isinstance(result, ClavaDDPMModelArtifacts)

    log(INFO, f"Result size (in bytes): {results_file.stat().st_size}")


INFO :      Checking the results from /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/tabular_data/multi_table/results/models/None_district_ckpt.pkl...
INFO :      Result size (in bytes): 2000325
INFO :      Checking the results from /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/tabular_data/multi_table/results/models/district_client_ckpt.pkl...
INFO :      Result size (in bytes): 20749428
INFO :      Checking the results from /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/tabular_data/multi_table/results/models/district_account_ckpt.pkl...
INFO :      Result size (in bytes): 20761515
INFO :      Checking the results from /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/tabular_data/multi_table/results/models/client_disp_ckpt.pkl...
INFO :      Result size (in bytes): 20649431
INFO :      Checking the results from /Users/fatemehtavakoli/D